librerías y ruta

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from pathlib import Path

PROYECTO_DIR = Path.cwd().resolve() 
DATOS_DIR = PROYECTO_DIR / "Datos"

Lista de activos y descarga de 2004-2025

In [2]:
activos = [
    # USA SP500 Nasdaq y Small Caps
    "^GSPC", "^NDX", "IWM",
    # Emerging Markets
    "EEM",
    # España y Europa
    "EWP", "FEZ",
    # Japón
    "EWJ",
    # Commodities oro y plata
    "GC=F", "SI=F",
    # Renta Fija
    "AGG", "TLT", "SHY",
    # Risk-free yield
    "^IRX"
]

start = "2004-01-01"
end   = "2026-01-01"

data_completa_2004_2025 = yf.download(activos, start=start, end=end, auto_adjust=False, progress=False)

precios = data_completa_2004_2025["Adj Close"].copy()
precios.index = pd.to_datetime(precios.index).normalize()
precios = precios.sort_index()
precios = precios[~precios.index.duplicated(keep="first")]

precios.head(), precios.tail()

(Ticker            AGG        EEM        EWJ        EWP        FEZ        GC=F  \
 Date                                                                            
 2004-01-02  50.237984  12.069791  27.436583  12.482225  17.463745         NaN   
 2004-01-05  50.297363  12.493290  28.002281  12.721585  17.796152  424.399994   
 2004-01-06  50.524879  12.352120  27.747725  12.761480  17.890411  422.799988   
 2004-01-07  50.638618  12.380932  27.521433  12.686130  17.642355  421.899994   
 2004-01-08  50.623783  12.427752  27.747725  12.850136  17.994604  424.000000   
 
 Ticker            IWM        SHY   SI=F        TLT        ^GSPC   ^IRX  \
 Date                                                                     
 2004-01-02  41.874897  54.226273    NaN  40.717148  1108.479980  0.902   
 2004-01-05  42.364719  54.232861  6.233  40.611084  1122.219971  0.902   
 2004-01-06  42.308643  54.298752  6.316  41.059525  1123.670044  0.901   
 2004-01-07  42.757359  54.338280  6.251  41.2234

In [3]:
print("Filas:", len(precios))
print("Columnas:", precios.shape[1])
print("Rango fechas:", precios.index.min().date(), "->", precios.index.max().date())

print("\n% NaN por activo:")
nan_pct = precios.isna().mean() * 100
display(nan_pct.sort_values(ascending=False).round(2))

print("\nPrimera fecha válida por activo:")
display(precios.apply(lambda s: s.first_valid_index()).to_frame("first_valid"))

print("\nÚltima fecha válida por activo:")
display(precios.apply(lambda s: s.last_valid_index()).to_frame("last_valid"))

print("\nMovimiento diario máximo (%) por activo:")
ret_temp = precios.pct_change(fill_method=None)
display((ret_temp.abs().max() * 100).sort_values(ascending=False).round(2).to_frame("max_abs_move_%"))

Filas: 5539
Columnas: 13
Rango fechas: 2004-01-02 -> 2025-12-31

% NaN por activo:


Ticker
GC=F     0.22
SI=F     0.20
^IRX     0.16
EEM      0.07
AGG      0.07
FEZ      0.07
EWP      0.07
EWJ      0.07
IWM      0.07
SHY      0.07
TLT      0.07
^GSPC    0.07
^NDX     0.07
dtype: float64


Primera fecha válida por activo:


,first_valid
Ticker,
AGG,2004-01-02
EEM,2004-01-02
EWJ,2004-01-02
EWP,2004-01-02
FEZ,2004-01-02
GC=F,2004-01-05
IWM,2004-01-02
SHY,2004-01-02
SI=F,2004-01-05



Última fecha válida por activo:


,last_valid
Ticker,
AGG,2025-12-31
EEM,2025-12-31
EWJ,2025-12-31
EWP,2025-12-31
FEZ,2025-12-31
GC=F,2025-12-31
IWM,2025-12-31
SHY,2025-12-31
SI=F,2025-12-31



Movimiento diario máximo (%) por activo:


,max_abs_move_%
Ticker,
^IRX,1214.29
EEM,22.77
SI=F,17.75
FEZ,17.53
EWP,16.29
EWJ,15.82
IWM,13.27
^NDX,12.58
^GSPC,11.98


In [4]:
first_valid = precios.apply(lambda s: s.first_valid_index())

fecha_inicio_real = first_valid.max()
print("Fecha inicio común:", fecha_inicio_real)

last_valid = precios.apply(lambda s: s.last_valid_index())

fecha_fin_real = last_valid.min()
print("Fecha fin común:", fecha_fin_real)

precios = precios.loc[(precios.index >= fecha_inicio_real) & (precios.index <= fecha_fin_real)]

Fecha inicio común: 2004-01-05 00:00:00
Fecha fin común: 2025-12-31 00:00:00


In [5]:
print("Total NaN por activo:")
display((precios.isna().sum()).sort_values(ascending=False))

print("\n% NaN por activo:")
display((precios.isna().mean()*100).round(3).sort_values(ascending=False))

print("Filas con algún NaN:", precios.isna().any(axis=1).sum())

nan_blocks = precios.isna().astype(int)

for col in precios.columns:
    max_block = nan_blocks[col].groupby(
        (nan_blocks[col] != nan_blocks[col].shift()).cumsum()
    ).sum().max()
    print(f"{col}: bloque máximo NaN = {max_block}")

Total NaN por activo:


Ticker
GC=F     11
SI=F     10
^IRX      9
EEM       4
AGG       4
FEZ       4
EWP       4
EWJ       4
IWM       4
SHY       4
TLT       4
^GSPC     4
^NDX      4
dtype: int64


% NaN por activo:


Ticker
GC=F     0.199
SI=F     0.181
^IRX     0.163
EEM      0.072
AGG      0.072
FEZ      0.072
EWP      0.072
EWJ      0.072
IWM      0.072
SHY      0.072
TLT      0.072
^GSPC    0.072
^NDX     0.072
dtype: float64

Filas con algún NaN: 20
AGG: bloque máximo NaN = 1
EEM: bloque máximo NaN = 1
EWJ: bloque máximo NaN = 1
EWP: bloque máximo NaN = 1
FEZ: bloque máximo NaN = 1
GC=F: bloque máximo NaN = 1
IWM: bloque máximo NaN = 1
SHY: bloque máximo NaN = 1
SI=F: bloque máximo NaN = 1
TLT: bloque máximo NaN = 1
^GSPC: bloque máximo NaN = 1
^IRX: bloque máximo NaN = 1
^NDX: bloque máximo NaN = 1


Se han seleccionado activos que permitan formar una cartera diversificada y además que no tengan grandes bloques de NaNs, los NaN que hay ahora se deben a cierres de mercado con días/horas de diferencia que se desajustan, pero se puede solventar fácil, un bloque grande de NaN sería más perjudicial, implicaría que para un periodo se han perdido los datos.

In [6]:
precios = precios.sort_index()
precios = precios.ffill()

print("Total NaN por activo:")
display((precios.isna().sum()).sort_values(ascending=False))

print("\n% NaN por activo:")
display((precios.isna().mean()*100).round(3).sort_values(ascending=False))

print("Filas con algún NaN:", precios.isna().any(axis=1).sum())

print("ALgún activo con precio negativo?\n", (precios <= 0).sum())


Total NaN por activo:


Ticker
AGG      0
EEM      0
EWJ      0
EWP      0
FEZ      0
GC=F     0
IWM      0
SHY      0
SI=F     0
TLT      0
^GSPC    0
^IRX     0
^NDX     0
dtype: int64


% NaN por activo:


Ticker
AGG      0.0
EEM      0.0
EWJ      0.0
EWP      0.0
FEZ      0.0
GC=F     0.0
IWM      0.0
SHY      0.0
SI=F     0.0
TLT      0.0
^GSPC    0.0
^IRX     0.0
^NDX     0.0
dtype: float64

Filas con algún NaN: 0
ALgún activo con precio negativo?
 Ticker
AGG      0
EEM      0
EWJ      0
EWP      0
FEZ      0
GC=F     0
IWM      0
SHY      0
SI=F     0
TLT      0
^GSPC    0
^IRX     7
^NDX     0
dtype: int64


Ya no hay NaN y el único activo con "precio" negativo es IRX, pero esto no es precio es un yield anual, por lo que puede ser negativo (tipos de interés negativos) es el único yield y luego veremos que lo separamos del resto para usarlo como activo libre de riesgo rf.

In [7]:
# separar IRX
irx_yield = precios["^IRX"].copy()
precios_activos = precios.drop(columns=["^IRX"])

# retornos pct:change calcula la varación con respecto al día anterior
retornos = precios_activos.pct_change(fill_method=None)

# eliminar primera fila ya que no puede comparar con el día anterior
retornos = retornos.dropna(how="any")

In [8]:
print("Rango retornos:")
print(retornos.index.min().date(), "→", retornos.index.max().date())
print("Número de días:", len(retornos))

print("NaN totales:", retornos.isna().sum().sum())

resumen_extremos = pd.DataFrame({
    "min": retornos.min(),
    "max": retornos.max()
})

display((resumen_extremos * 100).round(2).sort_values("min"))

Rango retornos:
2004-01-06 → 2025-12-31
Número de días: 5537
NaN totales: 0


,min,max
Ticker,,
SI=F,-17.75,12.97
EWP,-16.29,14.55
EEM,-16.17,22.77
IWM,-13.27,9.15
FEZ,-12.46,17.53
^NDX,-12.19,12.58
^GSPC,-11.98,11.58
EWJ,-10.41,15.82
GC=F,-9.35,9.03


No se ve ningún dato demasiado desorbitado (+200% en un día)

Vamos a construir ahora rf

In [9]:
# IRX es un yield anual, por lo que está en porcentaje anual
rf_anual = irx_yield.loc[retornos.index] / 100.0

print("Rango rf anual:")
print(rf_anual.min(), "→", rf_anual.max())

rf_diario = (1.0 + rf_anual) ** (1.0 / 252.0) - 1.0

print("Rango rf diario:")
print(rf_diario.min(), "→", rf_diario.max())

print("NaN en rf_anual:", rf_anual.isna().sum())
print("NaN en rf_diario:", rf_diario.isna().sum())


Rango rf anual:
-0.0010499999672174453 → 0.053480000495910646
Rango rf diario:
-4.168846879260002e-06 → 0.00020676331788527236
NaN en rf_anual: 0
NaN en rf_diario: 0


Rf tiene sentido
Vamos a hacer las features
Las features son variables que se derivan de los retornos históricos, aportan información relevante par nuestro futuro modelo, como momentum(tendencia) y volatilidad(riesgo). Se usan ventanas de 20 y 60 días para calcularlas, por eso perdemos los primeros 60 días. Además se desplazan un día hacia delante para no obtener estas variables con información actual y tener por tanto data leakage.

In [10]:
def construir_features(ret):
    ret = ret.sort_index().copy()

    momentum_20 = ret.rolling(20).sum().add_suffix("_mom20")
    momentum_60 = ret.rolling(60).sum().add_suffix("_mom60")

    volatilidad_20 = ret.rolling(20).std(ddof=1).add_suffix("_vol20")
    volatilidad_60 = ret.rolling(60).std(ddof=1).add_suffix("_vol60")

    feats = pd.concat([momentum_20, momentum_60, volatilidad_20, volatilidad_60], axis=1)

    #evitar leakage
    feats = feats.shift(1)
    feats = feats.dropna()

    return feats

features = construir_features(retornos)

In [11]:
print("Rango features:", features.index.min(), "→", features.index.max())
print("Fechas iguales retornos:", features.index.equals(retornos.loc[features.index].index))
print("NaNs features:", features.isna().sum().sum())

Rango features: 2004-04-01 00:00:00 → 2025-12-31 00:00:00
Fechas iguales retornos: True
NaNs features: 0


Para enriquecer el estado del agente, se incorporan también los retornos del día anterior junto con las features (momentum y volatilidad). Esto permite al modelo disponer de información inmediata sobre el comportamiento reciente del mercado, además de las dinámicas suavizadas capturadas por las ventanas móviles. Los retornos se desplazan un día (lag 1) para evitar data leakage, garantizando que en cada instante solo se utilice información disponible antes de tomar la decisión de inversión.

In [17]:
# retornos retrasados 1 día para evitar leakage
retornos_lag1 = retornos.shift(1)
# alinear con features (features ya vienen con shift(1) + dropna)
retornos_lag1 = retornos_lag1.loc[features.index]
# estado exógeno final
datos_estado = pd.concat([features, retornos_lag1], axis=1)
datos_estado = datos_estado.dropna()

# 2) alineación final obligatoria evita data leakage
fechas = (datos_estado.index.intersection(retornos.index).intersection(rf_diario.index))

datos_estado = datos_estado.loc[fechas]
retornos_ok = retornos.loc[fechas]
rf_ok = rf_diario.loc[fechas]
rf_anual_ok = rf_anual.loc[fechas]

print("datos_estado:", datos_estado.shape)
print("retornos_ok:", retornos_ok.shape)
print("rf_ok:", rf_ok.shape)
print("rf_anual_ok:", rf_anual_ok.shape)
print("Fechas iguales:",
      datos_estado.index.equals(retornos_ok.index) and
      datos_estado.index.equals(rf_ok.index))

datos_estado: (5477, 60)
retornos_ok: (5477, 12)
rf_ok: (5477,)
rf_anual_ok: (5477,)
Fechas iguales: True


Esto es lo que queremos:
- datos_estado: (5477, 60) → 60 variables de estado exógeno (features + retornos lag1)
- retornos_ok: (5477, 12) → 12 activos riesgosos para el reward
- rf_ok: (5477,) → rf diario y anual alineados
- Fechas iguales: True → alineación sin leakage por desfase

In [18]:
fecha_fin_train = "2016-01-01"
fecha_fin_validation = "2020-01-01"

# =========================
# ESTADO
# =========================
datos_estado_train = datos_estado.loc[datos_estado.index < fecha_fin_train]

datos_estado_validation = datos_estado.loc[
    (datos_estado.index >= fecha_fin_train)&(datos_estado.index < fecha_fin_validation)]

datos_estado_test = datos_estado.loc[datos_estado.index >= fecha_fin_validation]

# =========================
# RETORNOS (reward)
# =========================
retornos_train = retornos_ok.loc[datos_estado_train.index]
retornos_validation = retornos_ok.loc[datos_estado_validation.index]
retornos_test = retornos_ok.loc[datos_estado_test.index]

# =========================
# RISK FREE
# =========================
rf_train = rf_ok.loc[datos_estado_train.index]
rf_validation = rf_ok.loc[datos_estado_validation.index]
rf_test = rf_ok.loc[datos_estado_test.index]

rf_anual_train = rf_anual_ok.loc[datos_estado_train.index]
rf_anual_validation = rf_anual_ok.loc[datos_estado_validation.index]
rf_anual_test = rf_anual_ok.loc[datos_estado_test.index]

print("Train:", datos_estado_train.shape, retornos_train.shape, rf_train.shape, rf_anual_train.shape)
print("Validation:", datos_estado_validation.shape, retornos_validation.shape, rf_validation.shape, rf_anual_validation.shape)
print("Test:", datos_estado_test.shape, retornos_test.shape, rf_test.shape, rf_anual_test.shape)

Train: (2961, 60) (2961, 12) (2961,) (2961,)
Validation: (1006, 60) (1006, 12) (1006,) (1006,)
Test: (1510, 60) (1510, 12) (1510,) (1510,)


In [16]:
from entorno_cartera import EntornoCartera

entorno_train = EntornoCartera(datos_estado_train, retornos_train, rf_diario=rf_train)

estado_inicial = entorno_train.reset()
print("Dimensión estado entorno:", estado_inicial.shape)
print("Numero activos entorno:", entorno_train.numero_activos)

Dimensión estado entorno: (73,)
Numero activos entorno: 12


Lo guardamos todo en csv

In [20]:
CARPETA_DATOS = Path.cwd() / "Datos"   # si el notebook se ejecuta desde la raíz del proyecto
for sub in ["Raw", "Train", "Validation", "Test"]:
    (CARPETA_DATOS / sub).mkdir(parents=True, exist_ok=True)

# Guardar estado (features + retornos_lag1)
datos_estado.to_csv(CARPETA_DATOS / "Raw" / "datos_estado_full.csv")
datos_estado_train.to_csv(CARPETA_DATOS / "Train" / "datos_estado_train.csv")
datos_estado_validation.to_csv(CARPETA_DATOS / "Validation" / "datos_estado_validation.csv")
datos_estado_test.to_csv(CARPETA_DATOS / "Test" / "datos_estado_test.csv")

# Guardar retornos (reward)
retornos_ok.to_csv(CARPETA_DATOS / "Raw" / "retornos_full.csv")
retornos_train.to_csv(CARPETA_DATOS / "Train" / "retornos_train.csv")
retornos_validation.to_csv(CARPETA_DATOS / "Validation" / "retornos_validation.csv")
retornos_test.to_csv(CARPETA_DATOS / "Test" / "retornos_test.csv")

# Guardar rf diario
rf_ok.to_csv(CARPETA_DATOS / "Raw" / "rf_diario_full.csv")
rf_train.to_csv(CARPETA_DATOS / "Train" / "rf_diario_train.csv")
rf_validation.to_csv(CARPETA_DATOS / "Validation" / "rf_diario_validation.csv")
rf_test.to_csv(CARPETA_DATOS / "Test" / "rf_diario_test.csv")

rf_anual_ok.to_csv(CARPETA_DATOS / "Raw" / "rf_anual_full.csv")
rf_anual_train.to_csv(CARPETA_DATOS / "Train" / "rf_anual_train.csv")
rf_anual_validation.to_csv(CARPETA_DATOS / "Validation" / "rf_anual_validation.csv")
rf_anual_test.to_csv(CARPETA_DATOS / "Test" / "rf_anual_test.csv")

print("CSV guardados en:", CARPETA_DATOS.resolve())

CSV guardados en: C:\Users\inigo\Desktop\TFG Info\Datos
